# Evaluate SCALP-lite Embedding

Set `SCALP_INPUT_H5AD` to a file containing `adata.obsm["X_scalp"]`. Optionally set `SCALP_METRICS_CSV`.

In [ ]:
import os
from pathlib import Path

import seaborn as sns
import matplotlib.pyplot as plt

from scalp_lite import ScalpEstimator, score_embedding


def resolve_path(env_name, default):
    path = Path(os.environ.get(env_name, default))
    if path.exists() or path.is_absolute():
        return path
    notebook_relative = Path("..") / path
    return notebook_relative if notebook_relative.exists() else path


input_path = resolve_path("SCALP_INPUT_H5AD", "data/cellrank-pancreas-scalp.h5ad")
csv_path = Path(os.environ["SCALP_METRICS_CSV"]) if "SCALP_METRICS_CSV" in os.environ else input_path.parent / "scalp_lite_metrics.csv"
batch_key = os.environ.get("SCALP_BATCH_KEY", "batch")
label_key = os.environ.get("SCALP_LABEL_KEY", "label")

estimator = ScalpEstimator(batch_key=batch_key, label_key=label_key)


In [ ]:
adata = estimator.input(input_path)
scores = score_embedding(
    adata,
    embedding_key="X_scalp",
    batch_key=batch_key,
    label_key=label_key if label_key in adata.obs else None,
)
scores


In [ ]:
plot_df = scores.drop(columns=["embedding_key"], errors="ignore").melt(var_name="metric", value_name="value")
plt.figure(figsize=(9, 4))
sns.barplot(data=plot_df, x="metric", y="value")
plt.xticks(rotation=45, ha="right")
plt.tight_layout();

In [ ]:
scores.to_csv(csv_path, index=False)
csv_path